# Slew Rate

$$
\left|\frac{\partial U_{out}}{\partial t}\right| \leq v_s
$$

Ansatz $U_{out} = A\sin(\omega t), A, \omega > 0$. Damit $\frac{\partial U_{out}}{\partial t} = A\omega\cos(t)$ und wir definieren $U_{out}' = \int_0^t \max{\left(\min{(A\omega\cos(\omega\tau), -v_s)}, v_s\right)}d\tau$

Weiter definieren wir $A' = \frac{U_{out}'}{U_{out}}$. Es ergibt sich

$$
A'\approx\begin{cases} 
1 & \text{für } A \leq \frac{v_s}{\omega} \\
\frac{v_s}{A\omega} & \text{sonst}
\end{cases}
$$




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

v_s_raw = 0.3 / 1e-6   # uA741 
v_s     = v_s_raw / 1e-2
A       = 1.0

OMEGA_MIN = 2e3
OMEGA_MAX = v_s*1e9
SAMPLES_PER_CYCLE = 1024
N_CYCLES_DISPLAY = 3 


def cycle_grid(omega, n_cycles):
    return np.linspace(0, n_cycles * 2 * np.pi / omega, n_cycles * SAMPLES_PER_CYCLE)


def slew_limited_output(v_s_value, omega, t_values):
    dt = t_values[1] - t_values[0]
    dU = A * omega * np.cos(omega * t_values)
    return np.cumsum(np.clip(dU, -v_s_value, v_s_value)) * dt



def compute_a_prime_curve(v_s_value, t_frac, omega_values):
    u_out_at_t = A * np.sin(2 * np.pi * t_frac)

    if abs(u_out_at_t) < 1e-9:
        return np.full_like(omega_values, np.nan)

    a_prime = np.empty(len(omega_values))
    for i, w in enumerate(omega_values):
        t_abs  = t_frac * 2 * np.pi / w
        n      = max(int(t_frac * SAMPLES_PER_CYCLE), 32)
        t_vals = np.linspace(0, t_abs, n)
        u_prime_at_t = slew_limited_output(v_s_value, w, t_vals)[-1]
        a_prime[i]   = u_prime_at_t / u_out_at_t

    return a_prime

def compute_fft(v_s_value, omega):
    t     = cycle_grid(omega, 12)
    u_lim = slew_limited_output(v_s_value, omega, t)
    dt    = t[1] - t[0]
    half  = len(t) // 2
    sig   = u_lim[half:] - np.mean(u_lim[half:])
    spec  = np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(sig.size, d=dt)
    mag   = np.abs(spec) / sig.size
    return 2 * np.pi * freqs, mag


def update_plots(v_s_value, omega_value, t_frac):
    omega_c = v_s_value / A

    fig, axes = plt.subplots(1, 3, figsize=(21, 5))

    t1    = cycle_grid(omega_value, N_CYCLES_DISPLAY)
    u_lim = slew_limited_output(v_s_value, omega_value, t1)
    u_ref = A * np.sin(omega_value * t1)

    t_mark        = t_frac * 2 * np.pi / omega_value
    u_prime_mark  = np.interp(t_mark, t1, u_lim)
    u_ref_mark    = A * np.sin(omega_value * t_mark)

    axes[0].plot(t1, u_ref,  color="tab:gray", alpha=0.4, linestyle="--",
                 label=r"$U_{out}(t)$ (ideal)")
    axes[0].plot(t1, u_lim,  color="tab:blue",
                 label=r"$U_{out}'(t)$ (slew limited)")
    axes[0].axvline(t_mark, color="tab:red", linestyle=":", linewidth=1.5, alpha=0.8,
                    label=rf"$t = {t_frac:.2f}\,T$")
    axes[0].scatter([t_mark], [u_prime_mark], color="tab:blue",  zorder=5, s=70)
    axes[0].scatter([t_mark], [u_ref_mark],   color="tab:gray",  zorder=5, s=70)

    axes[0].set_title(r"$U_{out}'(t)$")
    axes[0].set_xlabel(r"$t$ (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=9)

    omegas       = np.geomspace(OMEGA_MIN, OMEGA_MAX, 500)
    a_prime_sim  = compute_a_prime_curve(v_s_value, t_frac, omegas)

    a_prime_approx = np.where(omegas <= omega_c, 1.0, omega_c / omegas)

    valid = ~np.isnan(a_prime_sim) & (a_prime_sim > 0)
    axes[1].plot(omegas[valid], a_prime_sim[valid],   color="tab:orange", linewidth=2,
                 label=rf"$A'(\omega;\;t={t_frac:.2f}\,T)$ (exact)")
    axes[1].plot(omegas,        a_prime_approx,        color="tab:red",    linewidth=1.5,
                 linestyle="--", alpha=0.7,
                 label=r"$\approx \min\!\left(1,\;\frac{v_s}{A\omega}\right)$ (approx.)")
    axes[1].axvline(omega_c,     color="gray",     linestyle=":", linewidth=1.2,
                    label=r"$\omega_c = v_s/A$")
    axes[1].axvline(omega_value, color="tab:blue", linestyle=":", linewidth=1.2, alpha=0.7,
                    label=r"current $\omega$")

    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlim(OMEGA_MIN, OMEGA_MAX)
    axes[1].set_ylim(5e-7, 2.0)
    axes[1].set_title(r"$A'(\omega;\;t)$ — changes with $t/T$ slider")
    axes[1].set_xlabel(r"$\omega$ (rad/s)")
    axes[1].set_ylabel(r"$A'$")
    axes[1].grid(alpha=0.3, which="both")
    axes[1].legend(fontsize=8)

    omega_fft, magnitude = compute_fft(v_s_value, omega_value)
    pos = omega_fft > 0
    axes[2].plot(omega_fft[pos], magnitude[pos], color="tab:green",
                 label=r"$|\mathcal{F}(U_{out}')|$")
    axes[2].set_xscale("log")
    axes[2].set_title(r"Fourier Transform of $U_{out}'$")
    axes[2].set_xlabel(r"$\omega$ (rad/s)")
    axes[2].set_ylabel("Magnitude")
    axes[2].grid(alpha=0.3, which="both")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

import math

v_s_slider = widgets.FloatLogSlider(
    value=v_s, base=10,
    min=math.log10(v_s * 0.1), max=math.log10(v_s * 3),
    step=0.01,
    description="v_s", readout_format=".3g",
)
omega_slider = widgets.FloatLogSlider(
    value=OMEGA_MIN, base=10,
    min=math.log10(OMEGA_MIN), max=math.log10(OMEGA_MAX),
    step=0.01,
    description="omega", readout_format=".3g",
)
t_slider = widgets.FloatSlider(
    value=0.25, min=0.02, max=0.48, step=0.01,
    description="t/T", readout_format=".2f",
)

controls = widgets.HBox([v_s_slider, omega_slider, t_slider])
out_all  = widgets.interactive_output(
    update_plots,
    {"v_s_value": v_s_slider, "omega_value": omega_slider, "t_frac": t_slider},
)

display(controls)
display(out_all)

Output()